<div align="center">

# Hafta 2: Makine öğrenmesi temelleri

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/lab.html) · [PDF ders notu](Hafta02_Ders_Notlari.pdf) · [Colab'da aç (yedek yol)](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/NB02_makine_ogrenmesi_temelleri.ipynb)

</div>

**Bu Jupyter defteri nasıl kullanılır?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Bölüm 0 ortamı hazırlar ve ilk önce çalıştırılmalıdır. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar, bunların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. Deftere ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır.

**Üç yol.** Etiketsiz hücrelerin tümü **temel yolu** oluşturur ve hiç kod yazmamış olanlar dâhil herkes için yeterlidir. **İleri yol** etiketli hücreler, programlama bilenler için isteğe bağlı genişletmelerdir. Üçüncü yolda kod yazmak yerine bir yapay zekâ (YZ) asistanına prompt (istem) yazılır: **Prompt yolunda** her alıştırmanın altında hazır promptu içeren katlanmış bir *Prompt yolu* hücresi bulunur ve her bölüm bir *Prompt görevi* ile biter. Promptu asistana kopyalayın, alıştırma hücresinin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın; sonucun doğru olup olmadığını denetim satırı gösterir. Haftanın [prompt kütüphanesi](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md) bütün promptları bir araya getirir; [prompt yolu rehberi](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/PROMPT_PATH.tr.md) ise bu defterle çalışan asistanları anlatmaktadır.

**Nerede çalışır?** Defter, kendi bilgisayarınızdaki Anaconda içinde Jupyter için hazırlanmıştır (ders klasöründeki SETUP.tr.md dosyasına bakın). Anaconda kurulamıyorsa aynı defter, en üstteki bağlantıyla Google Colab'da açılabilir. Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda ve kurumunuzun kurallarıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopya, örneğin Colab'da: Paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, precision_recall_curve,
                             roc_auc_score, roc_curve, silhouette_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from vclinic.datasets import load_cleveland
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 1, 0):
    raise ImportError("vclinic ders paketi bu defterden eski. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026

print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["ipywidgets", "lifelines"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 2.1": (6, 0),
    "Alıştırma 2.2": (76, 0),
    "Alıştırma 2.3": (0.855, 0.01),
    "Alıştırma A": (0.62, 0.01),
    "Alıştırma B": (0.931, 0.005),
    "Alıştırma C": (0.8, 0.001),
    "Alıştırma D": (0.871, 0.005),
    "Alıştırma E": (0.496, 0.001),
    "Alıştırma F": (0.46, 0.03),
    "Alıştırma G": (0.309, 0.001),
    "Alıştırma H": (0.194, 0.01),
    "Alıştırma I": (0.733, 0.002),
    "Alıştırma J": (0.849, 0.01),
})
print("Beklenen cevaplar hazır.")

## Python adımı 2: Veriyi modele hazırlamak

scikit-learn'deki bir model iki şey ister: Her satırı bir hasta olan ve `X` adı verilen bir sayı tablosu ile her hastanın sonucunu tutan `y`. Bu adım, 1. haftadaki Cleveland verisini böyle bir tabloya dönüştürür. Sütun ve satır seçer, eksik değerleri ele alır, bir kategoriyi sayılara çevirir, hastaları eğitim ve test kümelerine ayırır ve ilk modeli kurar. Defterin sonraki bütün bölümleri bu adımları kullanmaktadır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
heart = load_cleveland()
print(heart.shape)
heart[["age", "chol", "disease"]].head()

**Python notu.** `heart.shape` satır ve sütun sayısını verir. Çift köşeli parantez, `heart[["age", "chol"]]`, birden fazla sütun seçer ve daha küçük bir DataFrame döndürür. Tek köşeli parantez, `heart["age"]`, tek bir sütun seçer ve aynı satır etiketlerini taşıyan tek sütunluk bir *Series* döndürür.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
older = heart[heart["age"] >= 60]
print(len(older), "60 yaş ve üstü hasta")
heart.loc[heart["sex"] == 0, ["age", "chol", "disease"]].head(3)

**Python notu.** `heart["age"] >= 60` her yaşı 60 ile karşılaştırır ve `True` ile `False` değerlerinden oluşan bir sütun, yani bir *Boolean maske* verir. Maske köşeli parantez içine yazıldığında yalnızca `True` olan satırlar kalır. `.loc[satırlar, sütunlar]` satırları ve sütunları aynı anda seçer: Burada kadınlar (`sex == 0`) ve üç sütun.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
print(heart.isna().sum()[heart.isna().sum() > 0])
filled = heart["ca"].fillna(heart["ca"].median())
print("Doldurduktan sonra ca sütunundaki eksik:", filled.isna().sum())
print("Hiç eksik değeri olmayan satır:", len(heart.dropna()))

**Python notu.** Eksik bir değer `NaN` (not a number, sayı değil) olarak görünür. `isna()` onu işaretler, `.sum()` ise her sütundaki işaretleri sayar. `fillna(değer)` eksik değerlerin yerine bir değer koyar, burada sütunun ortancasını; `dropna()` ise eksik değeri olan her satırı siler. Bu haftanın modelindeki değişkenlerde eksik değer yoktur; bu yüzden defter 303 hastanın tamamını kullanır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
pd.get_dummies(heart["cp"], prefix="cp", drop_first=True, dtype=int).head()

**Python notu.** `cp` sütunu göğüs ağrısını kodlar: 1 tipik angina, 2 atipik angina, 3 anjinal olmayan ağrı, 4 ağrı yok. Buradaki sayılar miktar değil, addır: 4 numaralı ağrı, 2 numaralı ağrının iki katı değildir. Bu yüzden `pd.get_dummies` her tip için 0 ya da 1 değerini alan bir *kukla değişken* oluşturur. `drop_first=True` 1. tipi dışarıda bırakır ve onu referans yapar: Üç sütunun hepsinde 0 olan bir hastada tipik angina vardır.

In [ ]:
# Modelin tablosu: her satır bir hasta, yalnızca sayılar
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"],
                      drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
print("X:", X.shape, " eğitim kümesi:", X_train.shape, " test kümesi:", X_test.shape)
print("Hastalık oranı: eğitim", round(y_train.mean(), 3),
      " test", round(y_test.mean(), 3))

**Python notu.** `train_test_split` dört nesne döndürür; `X_train, X_test, y_train, y_test = ...` satırı bunları bu sırayla dört ada dağıtır. `test_size=0.25` hastaların dörtte birini ayrı tutar, `stratify=y` hastalığı olanların oranını iki parçada da aynı tutar, `random_state` ise ayırmanın her çalıştırmada aynı olmasını sağlar. Burada her hastanın bir satırı vardır. Bir hastanın birden fazla satırı olduğunda, örneğin birden fazla EKG'si bulunduğunda, ayırma bir hastanın bütün satırlarını aynı tarafta tutmalıdır; `GroupShuffleSplit` bunu hasta kimliğiyle yapar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
first = LogisticRegression(max_iter=1000)
first.fit(X_train, y_train)
print(first.predict(X_test[:5]))
print(first.predict_proba(X_test[:5]).round(3))

**Python notu.** scikit-learn'deki her model aynı kalıbı izler. `fit(X, y)` eğitim verisinden öğrenir. `predict(X)` 0 ya da 1 olan bir etiket döndürür; `predict_proba(X)` ise her sınıf için bir olasılık sütunu verir: İlk sütun 0'ın, ikinci sütun 1'in olasılığıdır. `[:5]` ilk beş satırı alır; `max_iter=1000` ise modelin uydurma işleminin yeterince uzun sürmesine izin verir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]
print("İlk beş test hastasının olasılıkları:", p_test[:5].round(3))

**Python notu.** `make_pipeline` adımları tek bir modelde zincirler. `StandardScaler` her sütunu eğitim verisinin ortalaması ve standart sapmasıyla bir *z skoruna*, $(x - \bar{x})/s$, dönüştürür. İşlem hattının içinde `fit` bu değerleri yalnızca eğitim hastalarından öğrenir ve test hastaları aynı sayılarla ölçeklenir; böylece test kümesinden modele hiçbir bilgi sızmaz. `[:, 1]` iki boyutlu bir dizinin bütün satırlarını (`:`) ve ikinci sütununu (`1`) alır.

**Hızlı kontrol.** Bir model `(76, 2)` boyutunda bir `predict_proba(X_test)` döndürüyor. 76 ve 2 neyi anlatır?

<details><summary>Cevap</summary>

76 satır, test kümesindeki 76 hastadır. 2 sütun 0 ve 1 sınıflarının olasılıklarıdır ve her satırda toplamları 1'dir.

</details>

**Alıştırma 2.1.** `heart` tablosunun bütün satır ve sütunlarında toplam kaç hücre eksiktir?

<details><summary><b>Prompt yolu</b>, Alıştırma 2.1: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `heart` adlı pandas DataFrame'i UCI Heart Disease veri setinin Cleveland verisini tutar: 303 hasta, her satırda bir hasta. disease sütunu 0 ya da 1'dir: Koroner arter hastalığı (KAH) varsa 1, yoksa 0. Eksik değerler NaN olarak durur.
Görev: `heart` tablosundaki bütün satır ve sütunlarda eksik olan hücrelerin toplamını say.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Eksik hücre sayısı, tamsayı olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 2.1", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Yöntemi önce `pd.DataFrame({"a": [1, None], "b": [None, 4]})` tablosunda dene; sonuç 2 olmalı. Ardından eksik değeri olan her sütunun eksik sayısını yazdır; listede yalnızca ca ve thal sütunları görünmeli.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 2.1", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = int(heart.isna().sum().sum())
teach.check("Alıştırma 2.1", answer)

**Alıştırma 2.2.** 303 hasta `test_size=0.25` ile ayrıldığında test kümesine kaç hasta düşer? Cevabı `len(...)` ile verin.

<details><summary><b>Prompt yolu</b>, Alıştırma 2.2: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X`, 303 satırlı ve 10 sayısal sütunlu bir DataFrame; `y` ise 0 ya da 1 olan 303 değerli bir Series.
Görev: `X` ve `y` verilerini `train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)` ile ayır ve test kümesindeki hastaları say.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Test kümesindeki hasta sayısı. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 2.2", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: İki kümenin büyüklüğünü yazdır; toplamları 303 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 2.2", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = len(X_test)
teach.check("Alıştırma 2.2", answer)

**Alıştırma 2.3.** `model` işlem hattının test kümesindeki doğruluğu, yani doğru etiketlenen hastaların oranı nedir? `model.score(X_test, y_test)` kullanın ve üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma 2.3: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X_train`, `X_test`, `y_train` ve `y_test`, `train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)` çağrısından gelir. `X` şu sütunları içerir: age (yıl), sex (1 erkek, 0 kadın), trestbps (istirahat sistolik kan basıncı, mmHg), chol (serum kolesterolü, mg/dL), thalach (en yüksek kalp hızı, bpm), exang (egzersizle angina, 0 ya da 1), oldpeak (ST çökmesi, mm) ve göğüs ağrısı tipinin üç kukla sütunu cp_2.0, cp_3.0 ve cp_4.0. `y`, disease sütunudur.
Görev: `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` kur, eğitim kümesinde uydur ve test kümesindeki doğruluğunu `score` ile hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Test kümesindeki doğruluk, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma 2.3", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Eğitim kümesindeki doğruluğu da yazdır; benzer büyüklükte olmalı ve 1,0 olmamalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma 2.3", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(model.score(X_test, y_test), 3)
teach.check("Alıştırma 2.3", answer)

## 1. İlk model: Koroner arter hastalığı için lojistik regresyon

*Ders: [Lojistik regresyon: Risk faktörlerinden bir olasılığa](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#logistic-regression-from-risk-factors-to-a-probability)*

Oturumun mini projesi, 303 Cleveland hastasının yedi değişkeninden ve göğüs ağrısı tipinden koroner arter hastalığı (KAH) olasılığını kestiren bir model kurar. Model eğitim kümesinde uydurulur ve bu bölümde incelenir. 2. bölüm onu test kümesinde ölçer.

**Formüllerden koda.** Doğrusal öngörücü $z = \beta_0 + \sum_j \beta_j x_j$, $\beta_0$ için `intercept_`, $\beta_j$ için `coef_` değerlerini kullanır. $p = 1/(1 + e^{-z})$ olasılığı, `predict_proba` çıktısının ikinci sütunudur. Bir değişkenin odds oranı $e^{\beta_j}$, yani `np.exp(coef_)` değeridir. `StandardScaler` sonrasında $x_j$ değerleri z skorudur; bu yüzden her odds oranı, değişkeninin bir standart sapmasına (SS) karşılık gelir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
# Modelin katsayıları: standart sapma başına odds oranları
lr = model.named_steps["logisticregression"]
coefs = pd.DataFrame({"beta": lr.coef_[0], "odds oranı": np.exp(lr.coef_[0])},
                     index=X.columns).sort_values("beta")
print("Sabit terim beta_0 =", round(lr.intercept_[0], 3))
coefs.round(3)

**Python notu.** `model.named_steps` bir işlem hattının adımlarını adlarıyla tutar; dolayısıyla `named_steps["logisticregression"]`, uydurulmuş lojistik regresyondur. `coef_[0]` onun ilk ve tek katsayı satırıdır. `sort_values("beta")` tabloyu en koruyucu değişkenden en zararlı değişkene doğru sıralar.

In [ ]:
# Haftanın vakası koşu bandında: z ve p elle ve modelle
case = pd.DataFrame([{"age": 57, "sex": 1, "trestbps": 119, "chol": 230, "thalach": 132,
                      "exang": 1, "oldpeak": 1.6, "cp_2.0": 1, "cp_3.0": 0, "cp_4.0": 0}])[X.columns]
scaler = model.named_steps["standardscaler"]
zscores = (case.to_numpy()[0] - scaler.mean_) / scaler.scale_
z = lr.intercept_[0] + np.sum(lr.coef_[0] * zscores)
p_hand = 1 / (1 + np.exp(-z))
print("z =", round(z, 3), " p (elle) =", round(p_hand, 3),
      " p (model) =", round(model.predict_proba(case)[0, 1], 3))

**Python notu.** `scaler.mean_` ve `scaler.scale_`, ölçekleyicinin eğitim kümesinden öğrendiği ortalama ve standart sapmadır. `case.to_numpy()[0]` tek satırlık tabloyu on sayılık bir diziye çevirir. `np.sum(a * b)` iki diziyi eleman eleman çarpar ve çarpımları toplar; bu da formüldeki $\sum_j \beta_j x_j$ toplamıdır.

**Alıştırma A.** VC-0246 numaralı hastada atipik angina vardır (`cp_2.0 = 1`). Ağrısı tipik angina olsaydı, yani üç `cp` sütununun hepsi 0 olsaydı, `model` işlem hattına göre KAH olasılığı ne olurdu? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma A: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `model`, KAH için uydurulmuş bir StandardScaler ve LogisticRegression işlem hattı; `case` ise `X` ile aynı sütunlara sahip tek satırlık bir DataFrame: age 57 yıl, sex 1, trestbps 119 mmHg, chol 230 mg/dL, thalach 132 bpm, exang 1, oldpeak 1,6 mm, cp_2.0 1, cp_3.0 0 ve cp_4.0 0.
Görev: `case` tablosunu kopyala, cp_2.0 değerini 0 yaparak üç göğüs ağrısı sütununu da 0'a getir (tipik angina) ve KAH olasılığını `model.predict_proba` ile hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Olasılık, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma A", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Değiştirilmemiş `case` için olasılığı da yazdır; iki olasılık farklı olmalı ve ikisi de 0 ile 1 arasında kalmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
typical = case.copy()
typical["cp_2.0"] = 0
answer = round(model.predict_proba(typical)[0, 1], 3)
teach.check("Alıştırma A", answer)

**Gözlemle ve düşün.** Cleveland verisinde göğüs ağrısı olmayan hastalarda (`cp_4.0 = 1`) hastalığın odds değeri en yüksektir; referans olan tipik anginada ise daha düşüktür. Hastaların sevk edilme biçimi bunu nasıl açıklayabilir? Bu durum, modelin farklı bir hasta karışımı gören bir klinikte kullanılması için ne anlama gelir?

**İleri yol.** Aynı işlem hattını `LogisticRegression(C=0.05, max_iter=1000)` ile uydurun. Küçük bir `C` güçlü bir *düzenlileştirme* anlamına gelir: Katsayılar sıfıra doğru küçülür. İki odds oranı tablosunu ve 2. bölümdeki test AUC değerini karşılaştırın.

In [ ]:
# İleri yol
strong = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=1000)).fit(X_train, y_train)
pd.DataFrame({"C=1": np.exp(lr.coef_[0]),
              "C=0.05": np.exp(strong.named_steps["logisticregression"].coef_[0])},
             index=X.columns).round(2)

**Prompt görevi, 1. bölüm: Odds oranlarının orman grafiği.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `model`, KAH için uydurulmuş bir StandardScaler ve LogisticRegression işlem hattı; `X`, sütunları modelin değişkenleri olan tablo.
Görev: Her değişkenin standart sapma başına odds oranını logaritmik yatay eksende bir nokta olarak çiz; 1'de dikey bir çizgi koy ve değişken adlarını dikey eksene yaz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Odds oranlarını da yazdır; thalach 1'in altında, oldpeak 1'in üstünde olmalı.
</pre>

</details>

**Beklenen sonuç.** SS başına odds oranı thalach için yaklaşık 0,58, oldpeak için yaklaşık 2,07'dir; sex ve cp_4.0 2'nin üstündedir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 2. Bir modeli ölçmek: Karışıklık matrisi ve ROC eğrisi

*Ders: [Bir modeli ölçmek: Karışıklık matrisi ve ROC eğrisi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#measuring-a-model-confusion-matrix-and-roc-curve)*

Test kümesinde modelin hiç görmediği 76 hasta vardır. Bir eşik her olasılığı bir karara dönüştürür; kararlar gerçek durumla karşılaştırılarak bir *karışıklık matrisinde* sayılır. Eşiği kaydırmak duyarlılık ile özgüllüğü birbirine karşı değiştirir; *alıcı işletim karakteristiği (ROC) eğrisi* ise bütün eşikleri aynı anda gösterir.

**Formüllerden koda.** `tp`, `fp`, `tn` ve `fn` sayılarıyla duyarlılık $Se = TP/(TP+FN)$, özgüllük $Sp = TN/(TN+FP)$, pozitif öngörü değeri $PPV = TP/(TP+FP)$, negatif öngörü değeri ise $NPV = TN/(TN+FN)$ olur. `roc_curve` yanlış pozitif oranını $1 - Sp$, doğru pozitif oranını $Se$ ve eşikleri; `roc_auc_score` ise eğri altındaki alanı (AUC) döndürür. Youden indeksi $J = Se + Sp - 1$ olarak tanımlanır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
# 0,5 eşiğinde karışıklık matrisi
pred = (p_test >= 0.5).astype(int)
print(pd.crosstab(y_test.to_numpy(), pred, rownames=["gerçek"], colnames=["model"]))
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")
print("Se =", round(tp / (tp + fn), 3), " Sp =", round(tn / (tn + fp), 3),
      " PPV =", round(tp / (tp + fp), 3), " NPV =", round(tn / (tn + fn), 3))

**Python notu.** `(p_test >= 0.5)` her hasta için `True` ya da `False` verir; `.astype(int)` bunları 1 ve 0'a çevirir. `pd.crosstab` gerçek durum ve karar çiftlerini sayar. `confusion_matrix(...).ravel()` dört sayıyı TN, FP, FN, TP sırasıyla döndürür; satır bunları dört ada dağıtır. Bir *f-string* süslü parantez içindeki değerleri metnin içine yerleştirir.

In [ ]:
# ROC eğrisi, AUC ve Youden indeksi
fpr, tpr, thresholds = roc_curve(y_test, p_test)
auc = roc_auc_score(y_test, p_test)
best = np.argmax(tpr - fpr)
print("AUC =", round(auc, 3))
print("Youden: eşik", round(thresholds[best], 3), " Se", round(tpr[best], 3),
      " Sp", round(1 - fpr[best], 3), " J", round(tpr[best] - fpr[best], 3))
fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot(fpr, tpr, color="#9B1B1B", lw=2, label=f"lojistik regresyon (AUC {auc:.3f})")
ax.plot([0, 1], [0, 1], color="#888888", ls="--", lw=1, label="şans")
ax.scatter(fpr[best], tpr[best], color="#1F5F8B", zorder=3, label="Youden")
ax.set_xlabel("1 − özgüllük (yanlış pozitif oranı)")
ax.set_ylabel("Duyarlılık (doğru pozitif oranı)")
ax.legend(loc="lower right")
plt.show()

**Python notu.** `roc_curve` işe yarayan bütün eşikleri dener ve aynı uzunlukta üç dizi döndürür. `np.argmax(tpr - fpr)`, $Se - (1 - Sp) = J$ değerinin en büyük olduğu konumu verir; bu konum da karşılık gelen eşiği seçer.

**Keşfet.** Eşiği kaydırın. Karışıklık matrisi, dört ölçüt ve ROC eğrisi üzerindeki nokta birlikte değişir. Düşük bir eşik kimseyi kaçırmaması gereken bir teste, yüksek bir eşik ise girişimsel bir işlemi ancak yeterince emin olduğunda başlatması gereken bir teste uyar.

In [ ]:
# Keşfet: eşik
def show_threshold(threshold=0.5):
    pred = (p_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    se, sp = tp / (tp + fn), tn / (tn + fp)
    ppv = tp / (tp + fp) if tp + fp else float("nan")
    fig, ax = plt.subplots(figsize=(4.4, 4.2))
    ax.plot(fpr, tpr, color="#9B1B1B", lw=2)
    ax.plot([0, 1], [0, 1], color="#888888", ls="--", lw=1)
    ax.scatter(1 - sp, se, s=60, color="#1F5F8B", zorder=3)
    ax.set_xlabel("1 − Sp")
    ax.set_ylabel("Se")
    ax.set_title(f"eşik {threshold:.2f}")
    plt.show()
    print(f"TP={tp} FP={fp} TN={tn} FN={fn} | Se={se:.3f} Sp={sp:.3f} PPV={ppv:.3f}")


try:
    from ipywidgets import interact, FloatSlider
    interact(show_threshold, threshold=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                                                   description="eşik"))
except ImportError:
    for value in (0.3, 0.5, 0.7):
        show_threshold(value)

**Alıştırma B.** `model` işlem hattının test kümesindeki AUC değeri nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma B: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `model`, KAH için uydurulmuş bir işlem hattı; test kümesi `X_test`, `y_test` (76 hasta; `y_test` 0 ya da 1'dir, KAH için 1).
Görev: `model` işlem hattının test kümesindeki olasılıklarının alıcı işletim karakteristiği eğrisi altındaki alanını (AUC) `roc_auc_score` ile hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: AUC, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma B", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 1 eksi olasılıkların AUC değerini de hesapla; iki değerin toplamı 1 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma B", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3)
teach.check("Alıştırma B", answer)

**Alıştırma C.** `model` işlem hattının test kümesinde 0,5 eşiğindeki duyarlılığı nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma C: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `model`, KAH için uydurulmuş bir işlem hattı; test kümesi `X_test`, `y_test` (76 hasta, 35'inde KAH var; `y_test` 0 ya da 1'dir).
Görev: KAH olasılığı 0,5 ya da üstündeyse hastayı pozitif say; TP, FP, TN ve FN sayılarını `confusion_matrix(...).ravel()` ile bul ve duyarlılığı TP / (TP + FN) ile hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Duyarlılık, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma C", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: TP ve FN değerlerini yazdır; toplamları, test kümesinde KAH olan hasta sayısı olan 35 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma C", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
pred = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
answer = round(tp / (tp + fn), 3)
teach.check("Alıştırma C", answer)

**Gözlemle ve düşün.** 0,93'lük bir AUC mükemmel görünür. Bir kardiyolog, modeli anjiyografi kararları için kullanmadan önce hâlâ hangi soruları sormalıdır? Test kümesinin büyüklüğünü, hastaların nereden geldiğini ve bir yanlış negatifin bedelini bir yanlış pozitifinkiyle karşılaştırarak düşünün.

**Prompt görevi, 2. bölüm: Yan yana üç eşik.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `p_test`, 76 test hastasının KAH olasılıkları; `y_test`, sonuçları.
Görev: 0,3, 0,5 ve 0,7 eşiklerinin her biri için bir satırı ve Se, Sp, PPV ve NPV sütunları olan bir tablo oluştur.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Üç ondalığa yuvarlanmış bir pandas DataFrame'i. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 0,5 eşiğinde tablo, 2. bölümdeki gibi Se 0,800 ve Sp 0,902 göstermeli.
</pre>

</details>

**Beklenen sonuç.** 0,3 ile 0,7 eşikleri arasında duyarlılık 0,857'den 0,571'e düşer, özgüllük 0,805'ten 0,951'e yükselir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 3. Çapraz doğrulama ve veri sızıntısı

*Ders: [Eğitim ve test: Çapraz doğrulama ve veri sızıntısı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#training-and-testing-cross-validation-and-data-leakage)*

76 hastalık tek bir test kümesi tek bir sayı verir; başka bir ayırma başka bir sayı verirdi. *K katlı çapraz doğrulama* hastaları $K$ parçaya böler, $K-1$ parçada eğitir, kalan parçada test eder ve bunu $K$ kez tekrarlar. $K$ sonucun ortalaması daha kararlı bir kestirimdir; sonuçların yayılımı ise bu kestirimin ne kadar oynadığını gösterir. Veriden öğrenen her adım, ölçekleme ve değişken seçimi dâhil, her katmanın içinde yeniden yapılmalıdır; aksi hâlde test hastalarının bilgisi modele *sızar*.

**Formüllerden koda.** Kestirim $\overline{AUC} = \frac{1}{K}\sum_{k=1}^{K} AUC_k$ olarak hesaplanır. `StratifiedKFold(5, shuffle=True, random_state=...)` her birinde hastalık oranı aynı olan beş katman oluşturur; `cross_val_score` ise beş $AUC_k$ değerini döndürür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
# Bütün işlem hattının beş katlı çapraz doğrulaması
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
print("Beş katmanın AUC değerleri:", scores.round(3))
print("Ortalama", round(scores.mean(), 3), " SS", round(scores.std(), 3))

**Python notu.** `cross_val_score` uydurulmamış işlem hattını ve `X` ve `y` verilerinin tamamını alır. Her katmanda işlem hattının yeni bir kopyasını dört parçada uydurur, böylece ölçekleyici de yalnızca eğitim katmanlarından öğrenir; ardından beşinci parçayı puanlar. `scoring="roc_auc"` AUC değerini ister.

In [ ]:
# Saf gürültüde sızıntı: 500 rastgele değişken ve hiçbir gerçek sinyal
rng = np.random.default_rng(RANDOM_STATE)
noise = rng.normal(size=(len(y), 500))

# Yanlış: 10 "en iyi" değişkeni bütün hastalarda seç, sonra çapraz doğrula
chosen = SelectKBest(f_classif, k=10).fit_transform(noise, y)
leaky = cross_val_score(LogisticRegression(max_iter=1000), chosen, y, cv=cv, scoring="roc_auc")

# Doğru: Seçim işlem hattının bir adımıdır ve her katmanda tekrarlanır
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression(max_iter=1000)),
                         noise, y, cv=cv, scoring="roc_auc")
print("Çapraz doğrulamadan önce seçim: AUC", round(leaky.mean(), 3))
print("İşlem hattı içinde seçim:          AUC", round(honest.mean(), 3))

**Python notu.** `rng.normal(size=(303, 500))` 303 satır ve 500 sütundan oluşan bir rastgele sayı tablosu üretir. `SelectKBest(f_classif, k=10)` iki grup arasında en çok farklılaşan 10 sütunu tutar. Saf gürültüde bazı sütunlar şans eseri farklılaşır; onları bütün hastalarla seçmek, test katmanlarını da seçime ortak eder.

**Alıştırma D.** `pipe` işlem hattının `cv` katmanları üzerindeki ortalama çapraz doğrulama AUC değeri nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma D: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X` ve `y`, 303 Cleveland hastası (`y` 0 ya da 1'dir, KAH için 1); `pipe`, StandardScaler ve LogisticRegression(max_iter=1000) adımlarından oluşan uydurulmamış bir işlem hattı.
Görev: `StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)` ve `cross_val_score(..., scoring="roc_auc")` ile beş katlı çapraz doğrulama yap ve beş AUC değerinin ortalamasını al.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Ortalama AUC, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma D", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Beş AUC değerini yazdır; her biri 0,5 ile 1 arasında olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma D", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean(), 3)
teach.check("Alıştırma D", answer)

**Gözlemle ve düşün.** Sızıntılı sürüm, hiçbir bilgi içermeyen veride yaklaşık 0,7'lik bir AUC bulmaktadır. Yayımlanmış modeller de bazen bu şekilde kurulur. Bir makalenin yöntem bölümündeki hangi cümle bu konuda sizi şüphelendirir, hangisi güven verir?

**İleri yol.** Çapraz doğrulamayı `sklearn.model_selection` içindeki `RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)` ile farklı karıştırmalarla on kez tekrarlayın ve 50 AUC değerinin histogramını çizin.

In [ ]:
# İleri yol
from sklearn.model_selection import RepeatedStratifiedKFold
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
many = cross_val_score(pipe, X, y, cv=rcv, scoring="roc_auc")
plt.hist(many, bins=12, color="#1F5F8B")
plt.xlabel("AUC")
plt.ylabel("Katman sayısı")
plt.show()
print(round(many.mean(), 3), round(many.std(), 3))

**Prompt görevi, 3. bölüm: Lojistik regresyon mu, rastgele orman mı?.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X` ve `y`, 303 Cleveland hastası; katmanlar `cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)`.
Görev: StandardScaler ve LogisticRegression(max_iter=1000) işlem hattının ortalama çapraz doğrulanmış AUC değerini aynı katmanlarda `RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)` ile karşılaştır.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: İki ortalama ve standart sapmaları. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Lojistik regresyon, Alıştırma D'deki ortalama AUC değerini vermeli.
</pre>

</details>

**Beklenen sonuç.** İki model de yaklaşık 0,87'lik bir ortalama AUC'ye ulaşır; bu 303 hastada rastgele orman daha iyi değildir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 4. Karar ağaçları ve rastgele ormanlar

*Ders: [Karar ağaçları ve rastgele ormanlar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#decision-trees-and-random-forests)*

Bir karar ağacı, klinik bir algoritma gibi her seferinde tek bir evet-hayır sorusu sorar. Her düğümde iki grubu en saf hâle getiren soruyu seçer. Rastgele orman ise her biri hastaların bootstrap örneğiyle ve her düğümde değişkenlerin rastgele bir alt kümesiyle kurulan yüzlerce ağaç üretir ve olasılıklarının ortalamasını alır.

**Formüllerden koda.** Sınıf oranları $p_0$ ve $p_1$ olan bir düğümün Gini safsızlığı $G = 1 - (p_0^2 + p_1^2)$ olarak tanımlanır. Bir bölünme, iki alt düğümün büyüklükleriyle ağırlıklandırılmış safsızlığıyla puanlanır. `DecisionTreeClassifier(max_depth=3)` ağacı üç soru düzeyiyle sınırlar; `RandomForestClassifier(n_estimators=300)` ise 300 ağacın ortalamasını alır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
# Üç soru düzeyli bir ağaç
tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Ağacın test AUC değeri:", round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3))
fig, ax = plt.subplots(figsize=(13, 5.5))
plot_tree(tree, feature_names=list(X.columns), class_names=["KAH yok", "KAH"],
          filled=True, impurity=True, fontsize=8, ax=ax)
plt.show()

**Python notu.** `plot_tree` her düğümü sorusu, Gini safsızlığı, hasta sayısı (`samples`) ve her sınıfın sayısıyla (`value`) çizer. Bir düğümün sol dalı, sorunun doğru olduğu hastaları içerir.

**Alıştırma E.** Ağacın kökündeki, yani hiçbir soru sorulmadan önceki Gini safsızlığını `y_train` içindeki iki sınıfın sayılarından hesaplayın. Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma E: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `y_train`, 227 eğitim hastasının 0 ya da 1 olan sonucu.
Görev: p0 = n0 / (n0 + n1) ve p1 = n1 / (n0 + n1) olmak üzere 1 - (p0² + p1²) döndüren bir `gini(n0, n1)` fonksiyonu yaz ve `y_train` içindeki 0 ve 1 sayılarına uygula.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Gini safsızlığı, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma E", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: assert ifadeleriyle denetle: gini(50, 50) 0,5, gini(100, 0) ise 0,0 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma E", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
n0, n1 = (y_train == 0).sum(), (y_train == 1).sum()
p0, p1 = n0 / (n0 + n1), n1 / (n0 + n1)
answer = round(1 - (p0 ** 2 + p1 ** 2), 3)
teach.check("Alıştırma E", answer)

In [ ]:
# Rastgele orman ve değişkenlerinin permütasyon önemi
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Ormanın test AUC değeri:", round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3))
imp = permutation_importance(forest, X_test, y_test, scoring="roc_auc", n_repeats=20,
                             random_state=RANDOM_STATE)
order = np.argsort(imp.importances_mean)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(X.columns[order], imp.importances_mean[order], xerr=imp.importances_std[order], color="#1F5F8B")
ax.set_xlabel("Değişken karıştırıldığında AUC'deki düşüş")
plt.show()

**Python notu.** *Permütasyon önemi*, test kümesindeki sütunları tek tek karıştırır ve AUC'nin ne kadar düştüğünü ölçer. Büyük bir düşüş, ormanın o değişkene dayandığını gösterir. `np.argsort` değerleri sıralayacak konumları döndürür; böylece çubuklar en küçükten en büyüğe doğru dizilir.

**Gözlemle ve düşün.** Tek bir ağaç kolay okunur ama ormandan daha düşük bir AUC'ye ulaşır; orman ise isabetlidir ama zor okunur. Yatak başındaki bir karar için hangisini tercih edersiniz ve ormana güvenmeden önce neyi görmeniz gerekir?

**Prompt görevi, 4. bölüm: Bir ağaç ne kadar derin olmalı?.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X_train`, `X_test`, `y_train` ve `y_test`, `train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)` çağrısından gelir. `X` şu sütunları içerir: age (yıl), sex (1 erkek, 0 kadın), trestbps (istirahat sistolik kan basıncı, mmHg), chol (serum kolesterolü, mg/dL), thalach (en yüksek kalp hızı, bpm), exang (egzersizle angina, 0 ya da 1), oldpeak (ST çökmesi, mm) ve göğüs ağrısı tipinin üç kukla sütunu cp_2.0, cp_3.0 ve cp_4.0. `y`, disease sütunudur.
Görev: 1'den 8'e kadar her d için eğitim kümesinde `DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)` uydur ve test AUC değerini derinliğe karşı çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve sekiz AUC değeri. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 3 derinliği, 4. bölümdeki ağacın test AUC değerini vermeli.
</pre>

</details>

**Beklenen sonuç.** AUC yaklaşık 0,81 ile 3 derinliğinde en yüksek değerine ulaşır ve daha derin ağaçlarda 0,75'in altına düşer: aşırı öğrenmenin bir görüntüsü.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 5. Risk polikliniği: Kayıttaki koroner arter hastalığı

*Ders: [Sanal klinik, 2. aşama: Risk polikliniği açılıyor](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#the-virtual-clinic-stage-2-the-risk-outpatient-clinic-opens)*

Sanal klinik 2000 hastaya ulaşır ve risk polikliniğini açar. Ders, her hastanın koroner arter hastalığını `vclinic.TRUE_EFFECTS` içinde saklanan bilinen log-odds etkilerinden üretmiştir. Bu nedenle kayıtta uydurulan bir lojistik regresyonun bu değerlere yakın katsayılar bulması beklenir; gerçek veride kurulan bir model bunu asla gösteremez. `vclinic.add_followup`, 6. bölümden itibaren kullanılan ekokardiyografi ayrıntılarını ve izlem verilerini ekler.

**Formüllerden koda.** Değişkenler `TRUE_EFFECTS` ile aynı biçimde kodlanır: 60 yaşın üstündeki her 10 yıl, $(LDL - 130)/40$, $(HDL - 50)/15$ ve diğerleri için 0 ya da 1. `LogisticRegression(C=np.inf)` modeli düzenlileştirme olmadan uydurur; bu yüzden `coef_` değerleri $\beta_j$ katsayılarının en çok olabilirlik kestirimleridir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
# İzlem verileriyle kayıt: 2000 hasta
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
print(registry.shape)
print(registry["main_diagnosis"].value_counts().to_string())


def risk_table(df):
    """Değişkenler vclinic.TRUE_EFFECTS ile aynı biçimde kodlanır."""
    return pd.DataFrame({
        "age_per_10y": (df["age"] - 60) / 10,
        "male": (df["sex"] == "M").astype(int),
        "smoker": df["smoker"],
        "diabetes": df["diabetes"],
        "hypertension": df["hypertension"],
        "ldl_per_40": (df["ldl"] - 130) / 40,
        "hdl_per_15": (df["hdl"] - 50) / 15,
        "family_history": df["family_history"],
    })


Z = risk_table(registry)
cad_model = LogisticRegression(C=np.inf, max_iter=2000).fit(Z, registry["cad"])
truth = pd.Series(vclinic.TRUE_EFFECTS)
compare = pd.DataFrame({"öğrenilen": np.r_[cad_model.intercept_, cad_model.coef_[0]],
                        "gerçek": truth[["intercept"] + list(Z.columns)].to_numpy()},
                       index=["intercept"] + list(Z.columns))
compare.round(2)

**Python notu.** `def` satırının hemen altında üç tırnak içindeki metin, yani *docstring*, fonksiyonun ne yaptığını anlatır. `np.r_[a, b]` iki diziyi tek bir dizide birleştirir. `truth[["intercept"] + list(Z.columns)]` gerçek değerleri öğrenilenlerle aynı sıraya koyar; böylece tablonun iki sütunu satır satır karşılaştırılabilir.

In [ ]:
# Haftanın vakası VC-0246 risk polikliniğinde
row = registry[registry["patient_id"] == "VC-0246"]
z_true = truth["intercept"] + (risk_table(row).iloc[0] * truth[Z.columns]).sum()
print(row[["age", "sex", "smoker", "hypertension", "diabetes", "ldl", "hdl", "family_history"]].to_string(index=False))
print("KAH olasılığı, model:", round(cad_model.predict_proba(risk_table(row))[0, 1], 3),
      " gerçek:", round(1 / (1 + np.exp(-z_true)), 3))
print("Kayıttaki koroner anjiyografi (cad):", int(row["cad"].iloc[0]))

**Alıştırma F.** `cad_model` `smoker` için hangi log-odds katsayısını öğrenmektedir? İki ondalığa yuvarlayın ve `TRUE_EFFECTS["smoker"]` ile karşılaştırın.

<details><summary><b>Prompt yolu</b>, Alıştırma F: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `registry` adlı pandas DataFrame'i sanal kliniğin 2000 sentetik hastasını, her satırda bir hasta olacak biçimde tutar; `vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))` ile oluşturulmuştur. Sütunları arasında age (yıl), sex ("M" ya da "F"), smoker, diabetes, hypertension, family_history ve cad (0 ya da 1), ldl ve hdl (mg/dL), egfr (mL/dk/1,73 m²), lvef (%), ntprobnp (pg/mL), af (0 ya da 1), bmi (kg/m²), sbp (mmHg), main_diagnosis, followup_months (ay) ve event (ölüm ya da kalp yetersizliği nedeniyle yatış için 1, sansürlenmiş için 0) bulunur.
Görev: Değişkenleri `vclinic.TRUE_EFFECTS` ile aynı biçimde kodla: age_per_10y = (age - 60) / 10, sex "M" ise male = 1, smoker, diabetes, hypertension, ldl_per_40 = (ldl - 130) / 40, hdl_per_15 = (hdl - 50) / 15 ve family_history. cad sütununa `LogisticRegression(C=np.inf, max_iter=2000)` uydur ve smoker katsayısını oku.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: smoker katsayısı, iki ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma F", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Öğrenilen bütün katsayıları `vclinic.TRUE_EFFECTS` değerlerinin yanına yazdır; her biri gerçek değerinin yaklaşık 0,2 yakınında olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma F", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(cad_model.coef_[0][list(Z.columns).index("smoker")], 2)
teach.check("Alıştırma F", answer)

**Gözlemle ve düşün.** Model doğru model olduğu hâlde öğrenilen katsayılar gerçeğe yakındır ama ona eşit değildir. 200 hastayla ve 20000 hastayla bu farklar ne olurdu? VC-0246'nın olasılığı yaklaşık 0,8'dir ve anjiyogramı normaldir. Model onun için yanılmış mıdır?

**Prompt görevi, 5. bölüm: Vakanın riski yaşlara göre.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `registry` adlı pandas DataFrame'i sanal kliniğin 2000 sentetik hastasını, her satırda bir hasta olacak biçimde tutar; `vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))` ile oluşturulmuştur. Sütunları arasında age (yıl), sex ("M" ya da "F"), smoker, diabetes, hypertension, family_history ve cad (0 ya da 1), ldl ve hdl (mg/dL), egfr (mL/dk/1,73 m²), lvef (%), ntprobnp (pg/mL), af (0 ya da 1), bmi (kg/m²), sbp (mmHg), main_diagnosis, followup_months (ay) ve event (ölüm ya da kalp yetersizliği nedeniyle yatış için 1, sansürlenmiş için 0) bulunur. `cad_model`, `risk_table(registry)` tablosuna uydurulmuş bir LogisticRegression modelidir; `risk_table(df)` fonksiyonu değişkenleri `vclinic.TRUE_EFFECTS` ile aynı biçimde kodlar.
Görev: VC-0246 numaralı hastanın satırını al, yalnızca yaşı 40'tan 80'e birer yıl artırarak değiştir ve `cad_model` modelinin KAH olasılığını yaşa karşı çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Eğri, 57 yaşta 5. bölümün VC-0246 için yazdırdığı olasılıktan geçmeli.
</pre>

</details>

**Beklenen sonuç.** Eğri 40 yaşta yaklaşık 0,55'ten 80 yaşta yaklaşık 0,9'a yükselir ve 57 yaşta yaklaşık 0,76'dan geçer.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 6. Nadir sonuçlar: Dengesiz veri

*Ders: [Nadir sonuçlar: Dengesiz veri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#rare-outcomes-imbalanced-data)*

12 ay içinde ölüm ya da kalp yetersizliği nedeniyle hastaneye yatış, kayıttaki her 100 hastanın yaklaşık 3'ünde görülür. Bu kadar nadir bir sonuçta her zaman *olay yok* diyen bir model 100 hastanın 97'sinde haklı çıkar ama hiçbir işe yaramaz. Bu bölüm doğruluğun neden yanılttığını, sınıf ağırlıklarının modeli nadir sınıfa doğru nasıl kaydırdığını ve pozitif öngörü değerinin neden prevalansa bağlı olduğunu gösterir.

**Formüllerden koda.** Bayes teoremi, $\pi$ prevalansı için $PPV = \frac{Se \cdot \pi}{Se \cdot \pi + (1 - Sp)(1 - \pi)}$ sonucunu verir. `class_weight="balanced"` her sınıfa $n/(2 n_k)$ ağırlığını verir; böylece nadir sınıf, sık görülen sınıf kadar hesaba katılır. `precision_recall_curve` her eşikte PPV'yi (*kesinlik*) ve duyarlılığı (*recall*) döndürür; `average_precision_score` ise eğriyi tek sayıyla özetler.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
# Nadir sonuç: 12 ay içinde olay
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)
EVENT_FEATURES = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes"]
Xe = registry[EVENT_FEATURES].assign(ntprobnp=np.log(registry["ntprobnp"]))
ye = registry["event_12m"]
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    Xe, ye, test_size=0.3, stratify=ye, random_state=RANDOM_STATE)
print("Prevalans:", round(ye.mean(), 3), " test kümesindeki olay:", int(ye_test.sum()), "/", len(ye_test))
print("Her zaman olay yok demenin doğruluğu:", round((ye_test == 0).mean(), 3))

rows = []
for weight in [None, "balanced"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=weight, max_iter=1000)).fit(Xe_train, ye_train)
    p = m.predict_proba(Xe_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(ye_test, (p >= 0.5).astype(int)).ravel()
    rows.append({"class_weight": str(weight), "AUC": roc_auc_score(ye_test, p),
                 "AP": average_precision_score(ye_test, p), "doğruluk": (tp + tn) / len(ye_test),
                 "Se": tp / (tp + fn), "PPV": tp / (tp + fp) if tp + fp else np.nan})
pd.DataFrame(rows).round(3)

**Python notu.** `.assign(ntprobnp=...)` bir sütunu değiştirilmiş, burada logaritmasıyla değiştirilmiş, bir tablo kopyası döndürür; çünkü NT-proBNP birkaç büyüklük basamağına yayılır. `str(weight)` tablo için `None` değerini "None" metnine çevirir. `np.nan` hesaplanamayan bir değeri, burada hiç pozitif karar verilmediği için PPV'yi işaretler.

In [ ]:
# Kesinlik ve duyarlılık; prevalansa göre PPV
balanced = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)).fit(Xe_train, ye_train)
precision, recall, _ = precision_recall_curve(ye_test, balanced.predict_proba(Xe_test)[:, 1])
prev = np.linspace(0.005, 0.5, 200)
se, sp = 0.85, 0.90
ppv = se * prev / (se * prev + (1 - sp) * (1 - prev))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(recall, precision, color="#9B1B1B")
ax[0].axhline(ye_test.mean(), color="#888888", ls="--", label="prevalans")
ax[0].set_xlabel("Duyarlılık (recall)")
ax[0].set_ylabel("Kesinlik (PPV)")
ax[0].legend()
ax[1].plot(prev * 100, ppv, color="#1F5F8B")
ax[1].set_xlabel("Prevalans (%)")
ax[1].set_ylabel("PPV")
ax[1].set_title("Se 0,85, Sp 0,90")
plt.show()

**Python notu.** `np.linspace(0.005, 0.5, 200)` eşit aralıklı 200 prevalans değeri üretir. Ardından Bayes teoreminin formülü, NumPy dizilerde eleman eleman çalıştığı için hepsine aynı anda uygulanır. `_`, kullanılmayacak bir dönüş değerine, burada eşiklere, verilen addır.

**Alıştırma G.** Bir tarama modelinin duyarlılığı 0,85, özgüllüğü 0,90'dır. Prevalansın yüzde 5 olduğu bir toplulukta PPV nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma G: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: Duyarlılığı 0,85 ve özgüllüğü 0,90 olan bir tarama modeli; prevalansın 0,05 (%5) olduğu bir toplulukta kullanılmaktadır.
Görev: Bayes teoremiyle PPV = se × prev / (se × prev + (1 - sp) × (1 - prev)) döndüren bir `ppv(se, sp, prev)` fonksiyonu yaz ve uygula.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: PPV, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma G", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: assert ifadesiyle denetle: ppv(0.85, 0.85, 0.5) değeri 0,85 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma G", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
se, sp, prev = 0.85, 0.90, 0.05
answer = round(se * prev / (se * prev + (1 - sp) * (1 - prev)), 3)
teach.check("Alıştırma G", answer)

**Gözlemle ve düşün.** Sınıf ağırlıklarıyla model olayların yaklaşık üçte ikisini bulur, ama alarmlarının ancak yaklaşık onda biri gerçek bir olaydır. Böyle bir model, bir kalp yetersizliği polikliniğinin kontrol vizitlerini planlamak için yararlı olur mu? Hangi eşiği seçerdiniz ve bu seçime kimler katılmalıdır?

**Prompt görevi, 6. bölüm: İki kesinlik-duyarlılık eğrisi.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `Xe_train`, `Xe_test`, `ye_train` ve `ye_test`, 12 ay içindeki olayın eğitim ve test kümeleri (prevalans yaklaşık 0,034); sütunlar age, egfr, lvef, ntprobnp'nin logaritması, af ve diabetes.
Görev: StandardScaler ve LogisticRegression(max_iter=1000) işlem hattını bir kez sınıf ağırlığı olmadan, bir kez `class_weight="balanced"` ile uydur ve iki kesinlik-duyarlılık eğrisini prevalansta yatay bir çizgiyle birlikte tek bir şekilde çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil ve iki ortalama kesinlik değeri. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: İki ortalama kesinlik değeri 6. bölümdeki tablonun AP sütunuyla aynı olmalı.
</pre>

</details>

**Beklenen sonuç.** İki eğri neredeyse üst üste biner ve ortalama kesinlik yaklaşık 0,10'dur: Sınıf ağırlıkları hastaların sıralamasını değil, eşiği kaydırır.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 7. Alt grupları bulmak: Kalp yetersizliği olan hastalarda k-ortalamalar

*Ders: [Denetimsiz öğrenme: k-ortalamalar ve fenogruplar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#unsupervised-learning-k-means-and-phenogroups)*

Kayıtta düşük ejeksiyon fraksiyonlu (HFrEF) ya da korunmuş ejeksiyon fraksiyonlu (HFpEF) kalp yetersizliği olan yaklaşık 400 hasta bulunur. Denetimsiz öğrenme, hiçbir sonuç bilgisi olmadan gruplar arar. *k-ortalamalar* algoritması $k$ merkez yerleştirir, her hastayı en yakın merkeze atar, her merkezi kendi hastalarının ortalamasına taşır ve hiçbir şey değişmeyene kadar bunu tekrarlar. Ders, kalp yetersizliği olan her hastanın içine bir fenogrup gizlemiştir. Kümeleme bunu görmez; bölümün sonunda kümeler bu gizli fenogruplarla karşılaştırılır.

**Formüllerden koda.** k-ortalamalar küme içi kareler toplamını, $WCSS = \sum_{k}\sum_{i \in C_k} \lVert \mathbf{x}_i - \mu_k \rVert^2$, en küçük yapar; `KMeans` bu değeri `inertia_` olarak bildirir. Bir hastanın siluet değeri $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$ olarak hesaplanır; burada $a(i)$ kendi kümesine, $b(i)$ ise en yakın diğer kümeye olan ortalama uzaklıktır. `silhouette_score` bütün hastaların ortalamasını döndürür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
# Kalp yetersizliği olan hastaların standartlaştırılmış öznitelikleri
hf = registry[registry["main_diagnosis"].isin(["HFrEF", "HFpEF"])].copy()
HF_FEATURES = ["age", "bmi", "egfr", "lvef", "ntprobnp", "lavi", "e_e_prime", "lv_mass_index", "tapse_mm"]
X_hf = StandardScaler().fit_transform(hf[HF_FEATURES].assign(ntprobnp=np.log(hf["ntprobnp"])))
print(len(hf), "kalp yetersizliği hastası")

ks = range(2, 7)
wcss, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_hf, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(list(ks), wcss, "o-", color="#1F5F8B")
ax[0].set_xlabel("k")
ax[0].set_ylabel("WCSS")
ax[1].plot(list(ks), sil, "o-", color="#9B1B1B")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Ortalama siluet")
plt.show()
print(pd.DataFrame({"k": list(ks), "WCSS": np.round(wcss, 1), "siluet": np.round(sil, 3)}).to_string(index=False))

**Python notu.** `.isin([...])` değeri listedeki değerlerden biri olan satırları tutar. `range(2, 7)` 2'den 6'ya kadar olan sayıları verir; döngü her biri için bir k-ortalamalar modeli uydurur. `n_init=10` her modeli merkezlerin on farklı rastgele konumundan başlatır ve en iyi sonucu tutar; çünkü k-ortalamalar kötü bir çözümde takılıp kalabilir.

In [ ]:
# Üç küme, ilk iki temel bileşen üzerinde
km3 = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
hf["cluster"] = km3.labels_
pca = PCA(n_components=2).fit(X_hf)
pcs = pca.transform(X_hf)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for c, color in zip(range(3), ["#9B1B1B", "#1F5F8B", "#B97813"]):
    ax.scatter(pcs[hf["cluster"] == c, 0], pcs[hf["cluster"] == c, 1], s=14, color=color, label=f"küme {c}")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.0f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.0f}%)")
ax.legend()
plt.show()
hf.groupby("cluster")[HF_FEATURES + ["diabetes", "af"]].mean().round(1).T

**Python notu.** Temel bileşen analizi (PCA), hastaların en çok farklılaştığı yönleri bulur. `explained_variance_ratio_` her yönün taşıdığı toplam varyans payını verir. Grafik kümeleri yalnızca gösterir, onları oluşturmaz: k-ortalamalar dokuz değişkenin tamamıyla çalışmıştır.

**Alıştırma H.** `X_hf` üzerinde üç kümeli k-ortalamalar çözümünün ortalama siluet değeri nedir? `silhouette_score(X_hf, km3.labels_)` kullanın ve üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma H: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `X_hf`, kalp yetersizliği olan her hasta için bir satırı olan bir NumPy dizisi; yaş (yıl), lvef (%) ve tapse_mm (mm) gibi dokuz sütunu z skoruna standartlaştırılmıştır.
Görev: `X_hf` dizisine `KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)` uydur ve etiketlerinin ortalama siluet değerini `silhouette_score` ile hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Ortalama siluet değeri, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma H", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Her kümedeki hasta sayısını yazdır; üç sayının toplamı `X_hf` dizisinin satır sayısı olan 407'ye eşit olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma H", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
answer = round(silhouette_score(X_hf, km3.labels_), 3)
teach.check("Alıştırma H", answer)

In [ ]:
# Kliniğin gizli gerçeği: fenogruplar
print(pd.crosstab(hf["cluster"], hf["hf_phenogroup"]))
print("Düzeltilmiş Rand indeksi:", round(adjusted_rand_score(hf["hf_phenogroup"], hf["cluster"]), 3))

**Gözlemle ve düşün.** Ortalama siluet değeri iki kümede üç kümeye göre biraz daha yüksektir; yine de üç küme gizli fenogruplarla iyi örtüşür. Karşılaştırılacak gizli bir gerçeğin olmadığı gerçek bir çalışmada fenogrup sayısına hangi ölçüt karar vermelidir? Kümeleri tablodaki profillerine göre adlandırın.

**Prompt görevi, 7. bölüm: Kümelerin ısı haritası.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `hf`, kalp yetersizliği olan hastaların cluster sütunu (0, 1 ya da 2) içeren DataFrame'i; `HF_FEATURES`, dokuz kümeleme değişkeninin listesi.
Görev: Her kümede her değişkenin ortalamasını hesapla, her değişkeni üç küme boyunca z skoruna çevir ve tabloyu hücrelere değerleri yazılmış bir ısı haritası olarak çiz.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Bir şekil. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Ortalama lvef değeri en düşük olan küme, ortalama ntprobnp değeri en yüksek olan küme de olmalı.
</pre>

</details>

**Beklenen sonuç.** Kümelerden birinde ejeksiyon fraksiyonu düşük ve NT-proBNP en yüksektir (HFrEF); biri en yaşlı, sol atriyumu büyük ve TAPSE değeri düşük olandır; biri de en genç, BKİ ve eGFR değeri en yüksek olandır.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 8. Sağkalım analizi: Kaplan-Meier ve log-rank testi

*Ders: [Sağkalım analizi: Kaplan-Meier ve log-rank testi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#survival-analysis-kaplan-meier-and-the-log-rank-test)*

İzlem farklı hastalarda farklı zamanlarda sona erer. Son vizitte hâlâ olaysız olan bir hasta *sansürlenir*: Olay daha sonra gelebilir ama görülmemiştir. Kaplan-Meier kestiricisi her hastayı izlendiği süre boyunca kullanır. Log-rank testi ise iki ya da daha fazla eğrinin şansın açıklayabileceğinden daha fazla farklılaşıp farklılaşmadığını sorar.

**Formüllerden koda.** Olay görülen her $t_i$ zamanında $n_i$ hasta risk altındadır ve $d_i$ hastada olay olur. Kestirim $S(t) = \prod_{t_i \le t} \left(1 - \frac{d_i}{n_i}\right)$ olarak hesaplanır. Log-rank istatistiği $X^2 = \sum_g \frac{(O_g - E_g)^2}{E_g}$ olarak tanımlanır; burada $O_g$, $g$ grubunda gözlenen olay sayısı, $E_g = \sum_i d_i\, n_{gi}/n_i$ ise beklenen olay sayısıdır. $G$ grup için istatistik, $G - 1$ serbestlik dereceli bir ki-kare dağılımıyla karşılaştırılır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
# Elle Kaplan-Meier
def kaplan_meier(time, event):
    """Basamakların zamanları ve her basamaktan sonraki sağkalım; (0, 1) noktasından başlar."""
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    times, surv, s = [0.0], [1.0], 1.0
    for t in np.unique(time[event == 1]):
        at_risk = np.sum(time >= t)
        events = np.sum((time == t) & (event == 1))
        s = s * (1 - events / at_risk)
        times.append(t)
        surv.append(s)
    return np.array(times), np.array(surv)


def survival_at(times, surv, t):
    """Basamak fonksiyonundan okunan S(t)."""
    return surv[np.searchsorted(times, t, side="right") - 1]


t10 = [2, 3, 3, 5, 6, 8, 9, 11, 12, 12]
e10 = [1, 1, 0, 1, 0, 1, 0, 1, 0, 0]
tt, ss = kaplan_meier(t10, e10)
pd.DataFrame({"ay": tt, "S(t)": ss.round(4)})

**Python notu.** `np.unique` olay görülen farklı zamanları sıralanmış olarak döndürür. `np.sum(time >= t)`, `t` zamanında hâlâ izlenen hastaları, yani $n_i$ değerini sayar; `(time == t) & (event == 1)` koşulu da o zamandaki olayları, yani $d_i$ değerini sayar. `np.searchsorted` bir zamanın eğrinin basamakları arasında nereye düştüğünü bulur.

In [ ]:
# Kümelere ve kalp yetersizliği tipine göre Kaplan-Meier eğrileri
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True)
for c, color in zip(range(3), ["#9B1B1B", "#1F5F8B", "#B97813"]):
    d = hf[hf["cluster"] == c]
    tt, ss = kaplan_meier(d["followup_months"], d["event"])
    ax[0].step(tt, ss, where="post", color=color, label=f"küme {c} (n={len(d)})")
for kind, color in zip(["HFrEF", "HFpEF"], ["#9B1B1B", "#1F5F8B"]):
    d = hf[hf["main_diagnosis"] == kind]
    tt, ss = kaplan_meier(d["followup_months"], d["event"])
    ax[1].step(tt, ss, where="post", color=color, label=f"{kind} (n={len(d)})")
    print(kind, "S(24) =", round(survival_at(tt, ss, 24), 3))
for a in ax:
    a.set_xlabel("Ay")
    a.legend(loc="lower left")
ax[0].set_ylabel("Ölüm ve yatış olmadan")
plt.show()


def logrank(time, event, group):
    """İki ya da daha fazla grup için log-rank testi: ki-kare ve p değeri."""
    time, event, group = np.asarray(time, float), np.asarray(event, int), np.asarray(group)
    labels = np.unique(group)
    observed = np.zeros(len(labels))
    expected = np.zeros(len(labels))
    for t in np.unique(time[event == 1]):
        at_risk = time >= t
        d = np.sum((time == t) & (event == 1))
        n = np.sum(at_risk)
        for j, g in enumerate(labels):
            observed[j] += np.sum((time == t) & (event == 1) & (group == g))
            expected[j] += d * np.sum(at_risk & (group == g)) / n
    chi2 = np.sum((observed - expected) ** 2 / expected)
    return chi2, stats.chi2.sf(chi2, len(labels) - 1)


for name, col in [("HFrEF ve HFpEF", "main_diagnosis"), ("kümeler", "cluster")]:
    chi2, p = logrank(hf["followup_months"], hf["event"], hf[col])
    print(f"{name}: X2 = {chi2:.2f}, p = {p:.2g}")

**Python notu.** `ax[0].step(..., where="post")` her değeri bir sonraki zamana kadar koruyan bir basamak fonksiyonu çizer; Kaplan-Meier eğrisi de böyle görünür. SciPy'deki `stats.chi2.sf(x, df)`, en az `x` kadar büyük bir ki-kare değerinin olasılığını, yani p değerini verir. `f"{p:.2g}"` bir sayıyı iki anlamlı basamakla yazdırır.

**Alıştırma I.** HFrEF hastalarında 24. ayda ölüm ya da hastaneye yatış olmadan geçen sürenin Kaplan-Meier kestirimi nedir? Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma I: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `hf`, kalp yetersizliği olan hastaların DataFrame'i; main_diagnosis ("HFrEF" ya da "HFpEF"), followup_months (ay) ve event (ölüm ya da yatış için 1, sansürlenmiş için 0) sütunlarını içerir.
Görev: Basamakların zamanlarını ve her basamaktan sonraki sağkalımı döndüren bir `kaplan_meier(time, event)` fonksiyonu yaz; S(t), t'ye kadarki olay zamanları üzerinden (1 - d_i / n_i) çarpımıdır. Fonksiyonu HFrEF hastalarına uygula ve 24. aydaki S değerini oku.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: S(24), üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma I", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 2, 3, 3, 5, 6, 8, 9, 11, 12, 12 aylık zamanlar ve 1, 1, 0, 1, 0, 1, 0, 1, 0, 0 olayları için 11. aydan sonraki sağkalım, dört ondalığa yuvarlandığında 0,3657 olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma I", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
d = hf[hf["main_diagnosis"] == "HFrEF"]
tt, ss = kaplan_meier(d["followup_months"], d["event"])
answer = round(survival_at(tt, ss, 24), 3)
teach.check("Alıştırma I", answer)

**Gözlemle ve düşün.** Bu kayıtta HFrEF ile HFpEF arasında belirgin bir fark yokken üç küme birbirinden ayrışmaktadır. Bu, hastaları yalnızca ejeksiyon fraksiyonuna göre gruplamak hakkında ne söyler? Hangi kümeyi daha yakından izlerdiniz ve bakımını değiştirmeden önce başka neyi bilmek isterdiniz?

**İleri yol.** `lifelines` paketi kuruluysa bu bölümdeki iki fonksiyonu `lifelines` içindeki `KaplanMeierFitter` ve `multivariate_logrank_test` ile karşılaştırın. `lifelines` paketinin log-rank istatistiği tam varyansı kullanır ve bu yüzden biraz daha büyüktür.

In [ ]:
# İleri yol
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import multivariate_logrank_test
    d = hf[hf["main_diagnosis"] == "HFrEF"]
    kmf = KaplanMeierFitter().fit(d["followup_months"], d["event"])
    print("lifelines S(24):", round(float(kmf.survival_function_at_times(24).iloc[0]), 3))
    print(multivariate_logrank_test(hf["followup_months"], hf["cluster"], hf["event"]).summary)
except ImportError:
    print("lifelines kurulu değil; bu bölümdeki fonksiyonlar eğitim için yeterlidir.")

**Prompt görevi, 8. bölüm: Risk altındaki hasta sayıları.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `hf`, kalp yetersizliği olan hastaların cluster, followup_months ve event sütunlarını içeren DataFrame'i.
Görev: Bir makalede Kaplan-Meier şeklinin altına yazılan biçimde, her küme için 0, 12, 24, 36 ve 48. aylarda risk altındaki, yani hâlâ izlenen ve olay yaşamamış hasta sayılarının tablosunu oluştur.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Her kümenin bir satır olduğu bir pandas DataFrame'i. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: 0. ayda sayılar kümelerin büyüklüklerine eşit olmalı.
</pre>

</details>

**Beklenen sonuç.** Sayılar her kümede azalır; en hızlı azalma en yaşlı hastaların kümesindedir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 9. YZ ile eşli kodlama, bölüm 2: Bir asistanın kurduğu modeli denetlemek

*Ders: [YZ ile eşli kodlama, bölüm 2: Bir asistanın kurduğu modeli denetlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html#ai-pair-programming-part-2-reviewing-a-model-built-by-an-assistant)*

Bir katılımcı bir YZ asistanından *kayıttaki verilerle 12 ay içinde ölümü ya da hastaneye yatışı öngören bir model ve onun AUC değerini* istemiştir. Aşağıdaki kod asistandan dönen cevaptır. Hatasız çalışır ve mükemmel bir sonuç bildirir. Çalıştırmadan önce dersteki model denetim sorularıyla okuyun: Model hangi değişkenleri görebilir ve hangi hastalarda ölçülmektedir?

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
# Bir YZ asistanının kodu: 12 aylık olayı öngör ve AUC değerini bildir
features = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "bmi", "sbp", "followup_months"]
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(registry[features], registry["event_12m"])
auc_reported = roc_auc_score(registry["event_12m"], rf.predict_proba(registry[features])[:, 1])
print("AUC:", round(auc_reported, 3))

**Gözlemle ve düşün.** Kodun iki sorunu vardır. Birincisi, karar anında modelin asla bilemeyeceği bir değişkendir. İkincisi AUC'nin ölçüldüğü yerdir. İkisini de adlandırın ve denetim sorularından hangisinin her birini yakaladığını söyleyin.

**Alıştırma J.** Kodu düzeltin: `followup_months` değişkenini çıkarın, `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)` kullanın ve `StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)` ile beş katlı çapraz doğrulamanın ortalama AUC değerini bildirin. Üç ondalığa yuvarlayın.

<details><summary><b>Prompt yolu</b>, Alıştırma J: Bir YZ asistanı için hazır prompt</summary>

Promptu bir YZ asistanına kopyalayın, bir sonraki hücrenin içeriğini asistanın döndürdüğü kodla değiştirin ve hücreyi çalıştırın. Kod, *doğru* ya da *henüz değil* yazan denetim satırıyla biter. *Henüz değil* yazarsa [prompt kütüphanesindeki](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/PROMPTLAR.md#denetimden-ge%C3%A7meyen-kodu-onarmak) onarım promptunu kullanın.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: `registry` adlı pandas DataFrame'i sanal kliniğin 2000 sentetik hastasını, her satırda bir hasta olacak biçimde tutar; `vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))` ile oluşturulmuştur. Sütunları arasında age (yıl), sex ("M" ya da "F"), smoker, diabetes, hypertension, family_history ve cad (0 ya da 1), ldl ve hdl (mg/dL), egfr (mL/dk/1,73 m²), lvef (%), ntprobnp (pg/mL), af (0 ya da 1), bmi (kg/m²), sbp (mmHg), main_diagnosis, followup_months (ay) ve event (ölüm ya da kalp yetersizliği nedeniyle yatış için 1, sansürlenmiş için 0) bulunur. event_12m sütunu 12 ay içinde olay varsa 1'dir. `features` listesi age, egfr, lvef, ntprobnp, af, diabetes, bmi, sbp ve followup_months sütunlarını tutar.
Görev: followup_months değişkenini çıkar, çünkü karar anında bilinmez. Ardından `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)` modelinin event_12m için ortalama AUC değerini beş katlı `StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)` ile hesapla.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Ortalama çapraz doğrulanmış AUC, üç ondalığa yuvarlanmış olarak. Sonucu `answer` adlı bir değişkende sakla ve kodu `teach.check("Alıştırma J", answer)` satırıyla bitir. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Aynı ormanın bütün hastalarda uydurulup yine onlarda puanlanan AUC değerini de yazdır; bu değer çapraz doğrulanmış değerden belirgin biçimde yüksek, 0,95'in üstünde olmalı.
</pre>

</details>

In [ ]:
answer = None   # cevabınız
teach.check("Alıştırma J", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
fixed = [f for f in features if f != "followup_months"]
forest_cv = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
answer = round(cross_val_score(forest_cv, registry[fixed], registry["event_12m"], cv=folds, scoring="roc_auc").mean(), 3)
teach.check("Alıştırma J", answer)

Etkileşimli labdaki [prompt atölyesi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/lab.html#part-f) böyle bir görevin promptunu prompt kartının yedi alanıyla yazar. *Test* alanı, iki sorunu da yakalayacak denetimin yeridir: modelin görmediği hastalarda sınanması.

**Prompt görevi, 9. bölüm: Asistanın modelinin denetimi.** Promptu bir YZ asistanına kopyalayın, asistanın döndürdüğü kodu aşağıdaki boş hücreye yapıştırın ve çalıştırın.

<details><summary>Bir YZ asistanı için <b>prompt</b></summary>

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Bir Jupyter defteri. NumPy np, pandas pd, matplotlib.pyplot plt, SciPy'nin stats modülü de stats adıyla içe aktarılmıştır; scikit-learn kuruludur. vclinic ders paketi ve teach modülü kullanılabilir; RANDOM_STATE = 2026'dır.
Veri: 9. bölümdeki YZ asistanı kodu, prompta metin olarak yapıştırılmış hâlde.
Görev: Kodu dört soruyla denetle: Hangi değişkenler karar anında bilinemez? Ayırmadan önce bütün hastalardan öğrenen bir adım var mı? AUC hangi hastalarda ölçülmektedir? Sonuç nadir mi? Ardından düzeltilmiş kodu ver.
Kısıtlar: Yalnızca bu paketleri ve burada adı geçen değişkenleri kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Sorunların kısa bir listesi ve düzeltilmiş kod. Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Düzeltilmiş kod, eğitim hastalarındaki AUC değerini değil, çapraz doğrulanmış AUC değerini bildirmeli.
</pre>

</details>

**Beklenen sonuç.** Denetim, izlem süresini gelecekten gelen bir bilgi olarak ve AUC'nin eğitim hastalarında ölçülmesini sorun olarak adlandırır; düzeltilmiş AUC 1,0 yerine yaklaşık 0,85'tir.

In [ ]:
# Prompt görevi: YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

## 10. Uygulama anahtarı: Başka bir sonuç, başka değişkenler

`OUTCOME` ve `VARIABLES` değerlerini değiştirin ve hücreyi yeniden çalıştırın. Hücre, bir lojistik regresyonun ve bir rastgele ormanın çapraz doğrulanmış AUC değerini ve lojistik regresyonun odds oranlarını bildirir. Denemeye değer sonuçlar: `"cad"`, `"event_12m"`, `"af"` ve `"event"`. Her biri için önce karar anında hangi değişkenlerin bilinebileceğini düşünün.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölüm kodlarını çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB02_makine_ogrenmesi_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
OUTCOME = "af"
VARIABLES = ["age", "bmi", "hypertension", "lvef", "lavi", "ntprobnp"]

Xo = registry[VARIABLES].copy()
if "ntprobnp" in Xo:
    Xo["ntprobnp"] = np.log(Xo["ntprobnp"])
yo = registry[OUTCOME]
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
lr_o = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
rf_o = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
print("Prevalans:", OUTCOME, round(yo.mean(), 3))
for name, m in [("lojistik regresyon", lr_o), ("rastgele orman", rf_o)]:
    s = cross_val_score(m, Xo, yo, cv=folds, scoring="roc_auc")
    print(f"{name}: AUC {s.mean():.3f} ± {s.std():.3f}")
lr_o.fit(Xo, yo)
pd.Series(np.exp(lr_o.named_steps["logisticregression"].coef_[0]), index=VARIABLES,
          name="SS başına odds oranı").round(2)

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Kalp yetersizliği | Kendi kliniğinizin protokolündeki izlem NT-proBNP değerini bir düşünce deneyi olarak ekleyin: Fenogrupları yatak başında daha kullanışlı kılmak için dokuz kümeleme değişkeninden hangisini değiştirirdiniz? Kendi seçiminizle yeniden kümeleyin ve Kaplan-Meier eğrilerini karşılaştırın. |
| Girişimsel kardiyoloji | 5. bölümdeki değişkenlerle kayıttan bir koroner arter hastalığı modeli kurun; hastalığı olmayan hastaların en fazla yüzde 20'sini anjiyografiye gönderecek eşiği seçin ve bu eşikteki duyarlılığı bildirin. |
| Elektrofizyoloji | Atriyal fibrilasyonu (`af`) yaş, BKİ, hipertansiyon, sol atriyum hacim indeksi ve NT-proBNP ile öngörün. Bilginin çoğunu hangi değişken taşımaktadır ve rastgele orman lojistik regresyona bir şey ekliyor mu? |
| Kardiyovasküler görüntüleme | Kalp yetersizliği olan hastaları yalnızca ekokardiyografik değişkenlerle (`lvef`, `lavi`, `e_e_prime`, `lv_mass_index`, `tapse_mm`) kümeleyin ve gizli fenogruplarla uyumu dokuz değişkenli çözümle karşılaştırın. |
| Koruyucu kardiyoloji | Diğer değerleri ortalama olan sigara içen ve içmeyen iki kişi için `cad_model` modelinin koroner arter hastalığı olasılığını yaşa karşı çizin. Bu modelde sigara, yaşa kaç yıl eklemektedir? |
| Kardiyo-onkoloji | Duyarlılığı 0,90 ve özgüllüğü 0,80 olan bir kardiyotoksisite tarama modeli düşünün. Yüzde 1 ile 20 arasındaki prevalanslar için PPV değerini çizin ve hangi prevalansta kullanılmaya değer olacağına karar verin. |
| Yoğun bakım | Sınıf ağırlıklı 12 aylık olay modelini kullanın ve olayların yüzde 80'ini bulan eşiği seçin. Model her 100 hastada kaç alarm verir ve bunların kaçı gerçek bir olaydır? |

## Defterden sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma metnini yazın ve öğrenme günlüğünü dışa aktarın. [Hafta sayfasında](README.md) anlatılan 2. haftanın haftalık projesi, 7. bölümdeki kümelerden ve 8. bölümdeki iki fonksiyondan başlar.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>